In [ ]:
CSV_FILE = "02dist.csv"

In [ ]:
from IPython.display import HTML
import matplotlib.animation as animation
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv(CSV_FILE)
df.columns = df.columns.str.strip()

timesteps = sorted(df["timestep"].unique())
nx = int(df["x"].max() + 1)
ny = int(df["y"].max() + 1)

D2Q9 = {
    0: (0, 0), 1: (1, 0), 2: (0, 1),
    3: (-1, 0), 4: (0, -1), 5: (1, 1),
    6: (-1, 1), 7: (-1, -1), 8: (1, -1),
}

num_timesteps = len(timesteps)
f = np.zeros((num_timesteps, nx, ny, 9))
t_map = {t: i for i, t in enumerate(timesteps)}

for _, row in df.iterrows():
    f[t_map[row["timestep"]], int(row["x"]), int(row["y"]), int(row["dir"])] = row["dist_value"]

cx = np.array([D2Q9[i][0] for i in range(9)])
cy = np.array([D2Q9[i][1] for i in range(9)])

rho = np.sum(f, axis=3)
rho_safe = np.where(rho == 0, 1.0, rho)
u_x_all = np.einsum("txy d, d -> txy", f, cx) / rho_safe
u_y_all = np.einsum("txy d, d -> txy", f, cy) / rho_safe

X, Y = np.meshgrid(np.arange(nx), np.arange(ny), indexing="ij")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.set_title(f"{CSV_FILE} — Velocity Vectors")
ax.set_xlabel("X"); ax.set_ylabel("Y")
ax.set_xlim(-1, nx); ax.set_ylim(-1, ny)

Q = ax.quiver(X, Y, u_x_all[0], u_y_all[0], color="blue", scale=15, pivot="mid")
time_text = ax.text(0.05, 0.93, "", transform=ax.transAxes, color="black", weight="bold")

def update(frame):
    Q.set_UVC(u_x_all[frame], u_y_all[frame])
    time_text.set_text(f"Timestep: {timesteps[frame]}")
    return Q, time_text

ani = animation.FuncAnimation(fig, update, frames=num_timesteps, interval=120, blit=True)
plt.close(fig)
HTML(ani.to_html5_video())

In [ ]:
x_1d = X[:, 0]
y_1d = Y[0, :]

fig, ax = plt.subplots(figsize=(8, 6))

def update_streamplot(frame):
    ax.clear()
    ax.set_title(f"{CSV_FILE} — Streamlines")
    ax.set_xlabel("X"); ax.set_ylabel("Y")
    ax.set_xlim(x_1d.min(), x_1d.max())
    ax.set_ylim(y_1d.min(), y_1d.max())

    u_x = u_x_all[frame].T
    u_y = u_y_all[frame].T
    ax.streamplot(x_1d, y_1d, u_x, u_y, color=np.sqrt(u_x**2 + u_y**2), cmap="viridis", density=1.2)
    ax.text(0.05, 0.93, f"Timestep: {timesteps[frame]}", transform=ax.transAxes,
            color="black", weight="bold")
    return []

ani_stream = animation.FuncAnimation(fig, update_streamplot, frames=num_timesteps, interval=150, blit=False)
plt.close(fig)
HTML(ani_stream.to_html5_video())